# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore the FAIR^2 dataset using the `mlcroissant` library, referencing all entities by their Croissant `@id`.

### Dataset Source
The dataset is defined by a Croissant schema at:
- [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # .to_json() is not necessary

print("Dataset name:", metadata.name)
print("\nDescription:")
print(metadata.description)


## 2. Data Overview
List available record sets, fields, and reference their Croissant `@id`.

This section discovers what data tables (record sets) and fields are available in the dataset.

In [ ]:
# List all record set @id and field @id for navigation and reference

recordsets = metadata.record_sets
print(f"Record sets in the dataset: {len(recordsets)}")
for idx, rs in enumerate(recordsets):
    print(f"[{idx}] RecordSet name: {rs.name}  (@id: {rs.id})")
    print(f"    Description: {rs.description}")
    if hasattr(rs, 'fields') and rs.fields:
        print(f"    Fields in this RecordSet:")
        for f in rs.fields:
            print(f"      - {f.name} (@id: {f.id}, dataType: {f.data_type})")
    print()


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All references are by `@id`.

We'll use the first tabular record set found.

In [ ]:
# Identify tabular record sets, select the first one for demonstration

# Use @id to reference each record set
record_set_ids = [rs.id for rs in recordsets]

if not record_set_ids:
    raise ValueError('No record sets found in dataset.')

print('Record set @id\'s:', record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from record set @id: {record_set_id}")

# Use the first non-empty record set for demonstration
main_record_set_id = None
for rsid, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rsid
        break

if main_record_set_id is None:
    raise ValueError('No non-empty record set found!')

print(f"Using record set: {main_record_set_id}")
print(f"Columns (field @id): {dataframes[main_record_set_id].columns.tolist()}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on a numeric field, normalizing, and grouping.

All fields are referenced via their Croissant `@id`.

In [ ]:
# Automatically pick a numeric field for demonstration

main_rs = None
for rs in metadata.record_sets:
    if rs.id == main_record_set_id:
        main_rs = rs
        break
if not main_rs:
    raise ValueError('Record set for EDA not found!')

numeric_field_id = None
numeric_field_type = None
for f in main_rs.fields:
    # Look for standard numeric types
    if f.data_type in ('Integer', 'Number', 'Float'):
        numeric_field_id = f.id
        numeric_field_type = f.data_type
        break

if not numeric_field_id:
    raise ValueError('No numeric field found in record set.')

print(f"Using numeric field @id: {numeric_field_id} (type: {numeric_field_type})")

# Filtering: e.g., threshold on value > 10
df = dataframes[main_record_set_id].copy()
numeric_series = pd.to_numeric(df[numeric_field_id], errors='coerce')
threshold = 10

filtered_df = df[numeric_series > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (total: {len(filtered_df)} records):")
print(filtered_df.head())

# Normalization of numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    numeric_series[filtered_df.index] - numeric_series[filtered_df.index].mean()
) / numeric_series[filtered_df.index].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by the first string/categorical field (if exists)
group_field_id = None
for f in main_rs.fields:
    if f.id != numeric_field_id and f.data_type in ('String', 'Text'):
        group_field_id = f.id
        break

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
    print(grouped_df.head())
else:
    print("No suitable string field found for grouping.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its normalized version.

You can explore relationships between fields using visualizations.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[numeric_field_id], kde=True, bins=30, color='skyblue')
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], kde=True, bins=30, color='salmon')
plt.title(f'Normalized {numeric_field_id} Distribution')
plt.xlabel(f"{numeric_field_id}_normalized")
plt.ylabel('Count')
plt.show()

# If grouping was possible, plot group mean
if group_field_id and group_field_id in filtered_df.columns:
    plt.figure(figsize=(10, 6))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
    plt.title(f'Mean {numeric_field_id} grouped by {group_field_id}')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- We explored the FAIR^2 dataset on mass spectrometry analysis of extracellular vesicles from human mast cells using Croissant `@id` references throughout.
- Fields and record sets were dynamically loaded and referenced by their schema identifiers.
- Common data processing and visualizations were demonstrated using pandas and seaborn/matplotlib.

**Next steps:** Explore more fields, correlate abundant proteins with experimental variables, or link with external protein annotation databases for deeper insight.